# 10 — Evidence-Grounded Prompting and RAG Interfaces

## Scenario
Northstar must answer questions about refund policies. We want to avoid hallucination, so we require the model to ground its answers in factual evidence.

**The Problem:** LLMs are eager to please and will often invent plausible-sounding policies if they don't know the answer.


In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab10 import CASES, build_grounded_request, build_requests, claim_is_supported, is_abstention, retrieve
from northstar.evidence import check_citations


def show_request(request):
    print("SYSTEM:\n", request.system)
    for message in request.messages:
        if message.text:
            print(f"{message.role.upper()}:\n{message.text}")
        for part in message.parts:
            print(f"{message.role.upper()} PART:", part)

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: Ungrounded Generation (Baseline)

Watch what happens when we ask a niche question without any grounding.


In [ ]:
request = next(r for r in build_requests() if r.case_id == "i10/ungrounded/custom-mug")
show_request(request)
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)
print("PARSED ANSWER:", response.text)


## Step 2: Manual Grounding (Classic RAG)

We "retrieve" a document (mocked here) and strictly instruct the model to use it.

Recorded-run observation: the model successfully restricted its answer to the supplied reference text.


In [ ]:
custom_case = next(case for case in CASES if case["id"] == "custom-mug")
evidence = retrieve(client, "custom mug Northstar logo", tenant="tenant-synthetic-a")
print("RETRIEVED:", evidence)
request = build_grounded_request(custom_case, evidence)
show_request(request)
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)
report = check_citations(response.text, evidence)
print("PARSED CITATION REPORT:", report)
assert report.unknown_ids == set()
assert "[POL-992]" in response.text
assert claim_is_supported(response.text, evidence, "final sale")


## Step 3: Managed Grounding with a Retrieval Tool (State of the Art)

Instead of manually pasting text into prompts, an application can expose a retrieval tool and keep evidence selection explicit. In replay mode this tool call and its response are deterministic; live mode can connect it to an approved provider or search service.


In [ ]:
tool_request = next(r for r in build_requests() if r.case_id == "i10/tool/custom-mug")
show_request(tool_request)
tool_response = client.generate(tool_request)
print("RECORDED TOOL CALL:", tool_response.tool_calls)
print("PARSED TOOL CALL:", tool_response.tool_calls[0].name)
assert tool_response.tool_calls[0].name == "search_policies"


### Retrieve the policy returned by the tool

The replayed tool call is followed by deterministic retrieval, so the evidence chain remains visible even offline.


In [ ]:
evidence = retrieve(client, "custom mug Northstar logo", tenant="tenant-synthetic-a")
print("RETRIEVED:", evidence)
assert any(item.id == "POL-992" for item in evidence)


### Ground the mug answer in `[POL-992]`

The application serializes the exact authorized retrieval result into the grounded request; the citation ID must exist and the answer must express the supported policy fact.


In [ ]:
answer_request = build_grounded_request(custom_case, evidence)
show_request(answer_request)
answer = client.generate(answer_request)
print("RECORDED GROUNDED ANSWER:", answer.text)
report = check_citations(answer.text, evidence)
print("PARSED CITATION REPORT:", report)
assert report.unknown_ids == set()
assert "[POL-992]" in answer.text
assert claim_is_supported(answer.text, evidence, "final sale")


### No-support abstention

When retrieval provides no supporting document, the recorded answer must abstain instead of inventing a policy.


In [ ]:
no_support_case = next(case for case in CASES if case["id"] == "no-support")
no_support_evidence = retrieve(client, no_support_case["question"], tenant="tenant-synthetic-a")
assert no_support_evidence == []
no_support_request = build_grounded_request(no_support_case, no_support_evidence)
show_request(no_support_request)
no_support = client.generate(no_support_request)
print("RECORDED ABSTENTION:", no_support.text)
print("PARSED ABSTENTION:", is_abstention(no_support.text))
assert is_abstention(no_support.text)


### Reject an unknown citation

A citation such as `[POL-404]` is rejected when it is not present in the retrieved evidence set.


In [ ]:
unknown_case = next(case for case in CASES if case["id"] == "unknown-citation")
unknown_request = build_grounded_request(unknown_case, evidence)
show_request(unknown_request)
unknown = client.generate(unknown_request)
unknown_report = check_citations(unknown.text, evidence)
print("RECORDED UNKNOWN-CITATION ANSWER:", unknown.text)
print("PARSED CITATION REPORT:", unknown_report)
assert "POL-404" in unknown_report.unknown_ids


## Takeaway
The recorded grounding run binds the generated request to the authorized retrieval result, cites `[POL-992]`, verifies the `final sale` fact against that evidence, rejects `[POL-404]` as an unknown ID, excludes a cross-tenant injected policy before ranking, and abstains 1/1 when retrieval finds no support.


## References
- [Core Concepts & Workflow](README.md#core-concepts--workflow)
- [Deep dive](README.md#deep-dive)
- [Lab walkthrough](README.md#lab-walkthrough)
